In [3]:
PREFIX = "https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/cohorts/2026/05-monitoring"

!wget -nc {PREFIX}/rag_helper.py
!wget -nc {PREFIX}/starter.py

File ‘rag_helper.py’ already there; not retrieving.



File ‘starter.py’ already there; not retrieving.



In [1]:
from opentelemetry import trace
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import ConsoleSpanExporter, SimpleSpanProcessor

# Capture spans (not just print them) so Q1/Q3 can be answered with real
# numbers instead of eyeballing console text -- ConsoleSpanExporter still
# does its normal printing via super().export(), this just also keeps a
# list and a count.
class CapturingConsoleExporter(ConsoleSpanExporter):
    def __init__(self):
        super().__init__()
        self.spans = []

    def export(self, spans):
        self.spans.extend(spans)
        return super().export(spans)


console_exporter = CapturingConsoleExporter()

provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(console_exporter))
trace.set_tracer_provider(provider)

tracer = trace.get_tracer("llm-zoomcamp")

In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
import starter

len(starter.documents)

72

In [4]:
from rag_helper import RAGBase

def calculate_cost(model, usage):
    """Same formula as 05-monitoring lesson 04 (metrics.py)."""
    cost = 0
    if "gpt-5.4-mini" in model:
        cost = (usage.input_tokens * 0.15 + usage.output_tokens * 0.60) / 1_000_000
    return cost

In [5]:
class RAGTraced(RAGBase):

    def rag(self, query):
        with tracer.start_as_current_span("rag"):
            return super().rag(query)

    def search(self, query, num_results=5):
        with tracer.start_as_current_span("search") as span:
            results = super().search(query, num_results=num_results)
            span.set_attribute("num_results", len(results))
            return results

    def llm(self, prompt):
        with tracer.start_as_current_span("llm") as span:
            response = super().llm(prompt)
            usage = response.usage
            span.set_attribute("input_tokens", usage.input_tokens)
            span.set_attribute("output_tokens", usage.output_tokens)
            span.set_attribute("cost", calculate_cost(self.model, usage))
            return response

In [6]:
assistant = RAGTraced(index=starter.index, llm_client=starter.client)

In [7]:
query = "How does the agentic loop keep calling the model until it stops?"
answer = assistant.rag(query)

print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0xbb07e5d400102046dd9b6f45a5cc2b8c",
        "span_id": "0x5e9da38b1953eff0",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xf84997371059dbaf",
    "start_time": "2026-07-18T06:40:20.830280Z",
    "end_time": "2026-07-18T06:40:20.956737Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "b81304ca-2fec-473d-add5-f0572fa9ffc8",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xbb07e5d400102046dd9b6f45a5cc2b8c",
        "span_id": "0xaa6cda6b7bd11971",
        "trace_state": "[]

In [8]:
# Q1: how many spans does the trace produce?
num_spans = len(console_exporter.spans)
print("Q1 - number of spans:", num_spans)

# Q3: span durations (nanoseconds -> ms)
for s in console_exporter.spans:
    duration_ms = (s.end_time - s.start_time) / 1_000_000
    print(f"Q3 - span '{s.name}' duration_ms: {duration_ms:.1f}")

Q1 - number of spans: 3
Q3 - span 'search' duration_ms: 126.5
Q3 - span 'llm' duration_ms: 9619.7
Q3 - span 'rag' duration_ms: 9749.6


In [9]:
llm_span = next(s for s in console_exporter.spans if s.name == "llm")
input_tokens = dict(llm_span.attributes)["input_tokens"]
print("Q2 - llm span input_tokens:", input_tokens)

Q2 - llm span input_tokens: 7111


In [10]:
import sqlite3
from opentelemetry.sdk.trace.export import SpanExporter, SpanExportResult


class SQLiteSpanExporter(SpanExporter):

    def __init__(self, db_path="traces.db"):
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
            CREATE TABLE IF NOT EXISTS spans (
                name TEXT,
                start_time INTEGER,
                end_time INTEGER,
                input_tokens INTEGER,
                output_tokens INTEGER,
                cost REAL
            )
        """)
        self.conn.commit()

    def export(self, spans):
        for span in spans:
            attrs = dict(span.attributes or {})
            self.conn.execute(
                "INSERT INTO spans VALUES (?, ?, ?, ?, ?, ?)",
                (
                    span.name,
                    span.start_time,
                    span.end_time,
                    attrs.get("input_tokens"),
                    attrs.get("output_tokens"),
                    attrs.get("cost"),
                ),
            )
        self.conn.commit()
        return SpanExportResult.SUCCESS

    def shutdown(self):
        self.conn.close()

    def force_flush(self):
        return True


# Added as a second processor alongside the console one (not a strict
# replace) -- spans keep printing to console for visibility, and now also
# land in traces.db. Functionally equivalent to "swap the destination"
# for everything from this point on, since nothing reads console_exporter
# after this cell.
#
# Start from a clean database file every time this cell runs. SQLiteSpanExporter
# only does CREATE TABLE IF NOT EXISTS, so re-running the notebook (kernel
# restart, re-running top to bottom) against a traces.db left over from a
# previous run would silently accumulate extra rows -- breaking Q6's "4 runs
# total" count and inflating Q5's per-span-name duration totals. Deleting the
# file first makes a full top-to-bottom run idempotent and keeps the row count
# matching what the questions actually assume.
import os

if os.path.exists("traces.db"):
    os.remove("traces.db")

provider.add_span_processor(SimpleSpanProcessor(SQLiteSpanExporter("traces.db")))

In [11]:
answer = assistant.rag(query)
print(answer)

{
    "name": "search",
    "context": {
        "trace_id": "0x8f801b80b0426e8a4987cea523620888",
        "span_id": "0x909222cca85b35db",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x47fa23d002a9ccbe",
    "start_time": "2026-07-18T06:41:58.623191Z",
    "end_time": "2026-07-18T06:41:58.658308Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "b81304ca-2fec-473d-add5-f0572fa9ffc8",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0x8f801b80b0426e8a4987cea523620888",
        "span_id": "0x17a6dc43e32d0178",
        "trace_state": "[]

In [12]:
import pandas as pd

df = pd.read_sql("SELECT * FROM spans", sqlite3.connect("traces.db"))
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784356918623191169,1784356918658308158,NaN,NaN,NaN
1,llm,1784356918691686246,1784356921296156556,7111.0,133.0,0.001146
2,rag,1784356918623050930,1784356921331224239,NaN,NaN,NaN


In [13]:
# Q4: which span names appear in the spans table?
span_names = sorted(df["name"].unique().tolist())
print("Q4 - span names in traces.db:", span_names)

# Q5: excluding rag, which span type takes the most total time?
df["duration_ms"] = (df["end_time"] - df["start_time"]) / 1_000_000
totals = df[df["name"] != "rag"].groupby("name")["duration_ms"].sum()
print("Q5 - total duration by span name (excluding rag):")
print(totals)

Q4 - span names in traces.db: ['llm', 'rag', 'search']
Q5 - total duration by span name (excluding rag):
name
llm       2604.470310
search      35.116989
Name: duration_ms, dtype: float64


In [14]:
# Q6: run the same query 3 more times (4 total RAG calls in the database,
# counting the one just above), then check input-token stability.
for i in range(3):
    assistant.rag(query)

df = pd.read_sql("SELECT * FROM spans WHERE name = 'llm'", sqlite3.connect("traces.db"))
print("Q6 - llm rows in traces.db:", len(df))
print("Q6 - input_tokens per run:", df["input_tokens"].tolist())

mean_tokens = df["input_tokens"].mean()
max_dev = (df["input_tokens"] - mean_tokens).abs().max()
pct_variance = (max_dev / mean_tokens) * 100 if mean_tokens else 0
print(f"Q6 - max deviation from mean: {pct_variance:.2f}%")

{
    "name": "search",
    "context": {
        "trace_id": "0xf126fdcd2a959ca9d3de64cb6b93d8ec",
        "span_id": "0xe00f8a3927708312",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0xc2363ec24ff0260c",
    "start_time": "2026-07-18T06:43:12.600834Z",
    "end_time": "2026-07-18T06:43:12.669118Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "num_results": 5
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.44.0",
            "service.instance.id": "b81304ca-2fec-473d-add5-f0572fa9ffc8",
            "service.name": "unknown_service"
        },
        "schema_url": ""
    }
}
{
    "name": "llm",
    "context": {
        "trace_id": "0xf126fdcd2a959ca9d3de64cb6b93d8ec",
        "span_id": "0x3a90d9661b9d5d3e",
        "trace_state": "[]

In [15]:
print("=== Answers summary ===")
print("Q1 (span count):        ", num_spans)
print("Q2 (llm input tokens):  ", input_tokens)
print("Q4 (span names):        ", span_names)
print("Q5 (duration by name):  ")
print(totals)
print("Q6 (llm input_tokens across 4 runs):", df["input_tokens"].tolist(), f"{pct_variance:.2f}%")

=== Answers summary ===
Q1 (span count):         3
Q2 (llm input tokens):   7111
Q4 (span names):         ['llm', 'rag', 'search']
Q5 (duration by name):  
name
llm       2604.470310
search      35.116989
Name: duration_ms, dtype: float64
Q6 (llm input_tokens across 4 runs): [7111, 7111, 7111, 7111] 0.00%
